# DLModel demo

Runs the full `DLModel` pipeline end-to-end on small dummy data (2 clusters × 50 parts × 60 months).

This notebook provides stand-alone stubs for the external helpers (`get_user_defined_score`, `ReverseSmooth`, `Postprocessing`, `month_format`) since they are not in this repo snippet. Swap in the real ones when running against production data.

In [ ]:
import numpy as np
import pandas as pd
import torch

np.random.seed(0)
torch.manual_seed(0)

import model_dl
print('Using device:', model_dl.device)

## 1. Stubs for external helpers

Inject these into `model_dl`'s namespace. Replace with the real implementations when running against production.

In [ ]:
month_format = '%Y-%m'

def get_user_defined_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2) + 1e-8
    return 1.0 - ss_res / ss_tot  # R^2, higher is better

def ReverseSmooth(gi_data, pred):
    # Stub: no smoothing to reverse. Join the tuple index into 'version|plant|part'.
    out = pred.copy()
    out.name = '|'.join(str(s) for s in pred.name)
    return out

def Postprocessing(df):
    # Stub: clip negatives to 0 (GI is non-negative).
    return df.clip(lower=0)

# Inject into model_dl's globals (same pattern as model.py's bare references).
model_dl.month_format = month_format
model_dl.get_user_defined_score = get_user_defined_score
model_dl.ReverseSmooth = ReverseSmooth
model_dl.Postprocessing = Postprocessing

from model_dl import DLModel

## 2. Dummy data

2 clusters, 50 parts each, 1 plant per part, 60 months of monthly GI and MOU (2021-03 through 2026-02). Validation month is `2026-03` (first forecast month, 6-month horizon).

In [ ]:
N_CLUSTERS = 2
N_PARTS_PER_CLUSTER = 50
N_MONTHS = 60
LAG_GI = 12
LAG_MOU = 6
HORIZON = 6
PLANT = '0020'
HISTORY_END = pd.Timestamp('2026-02-01')
VALIDATION_MONTH = '2026-03'
VALIDATION_COLUMNS = ['m0', 'm1', 'm2', 'm3', 'm4', 'm5']

months = pd.date_range(end=HISTORY_END, periods=N_MONTHS, freq='MS')

parts = [f'{c:02d}{p:04d}' for c in range(N_CLUSTERS) for p in range(N_PARTS_PER_CLUSTER)]
cluster_of = {f'{c:02d}{p:04d}': c for c in range(N_CLUSTERS) for p in range(N_PARTS_PER_CLUSTER)}

cluster_df = pd.DataFrame({'part': parts, 'cluster': [cluster_of[p] for p in parts]})

def simulate_gi(n_months, seed):
    rng = np.random.default_rng(seed)
    t = np.arange(n_months)
    trend = 0.05 * t
    season = 3.0 * np.sin(2 * np.pi * t / 12.0)
    noise = rng.normal(0, 1.0, size=n_months)
    base = rng.uniform(5, 15)
    gi = np.clip(base + trend + season + noise, 0, None)
    return gi

def simulate_mou(n_months, seed):
    rng = np.random.default_rng(seed + 10000)
    return np.clip(rng.normal(0.5, 0.15, size=n_months), 0, 1)

gi_records = []
mou_records = []
for i, part in enumerate(parts):
    gi_records.append(simulate_gi(N_MONTHS, seed=i))
    mou_records.append(simulate_mou(N_MONTHS, seed=i))

idx = pd.MultiIndex.from_tuples([(p, PLANT) for p in parts], names=['part', 'plant'])
gi_plant_df = pd.DataFrame(gi_records, index=idx, columns=months)
mou_plant_df = pd.DataFrame(mou_records, index=idx, columns=months)
print('GI shape:', gi_plant_df.shape, '| MOU shape:', mou_plant_df.shape)
gi_plant_df.head(3)

## 3. Build X_train, y_train, X_test, gi_selected per cluster

This directly produces the DataFrames that `DLModel` expects. In production, `getData` + `split_data` build these from the raw frames; we bypass them here for a self-contained demo.

In [ ]:
def months_offset(anchor, k):
    return (anchor + pd.DateOffset(months=k)).normalize()

def build_cluster_frames(cluster_id, gi_plant_df, mou_plant_df, cluster_df, months, test_month):
    cluster_parts = cluster_df.loc[cluster_df['cluster'] == cluster_id, 'part'].tolist()
    gi_sel = gi_plant_df.loc[gi_plant_df.index.get_level_values('part').isin(cluster_parts)]
    mou_sel = mou_plant_df.loc[mou_plant_df.index.get_level_values('part').isin(cluster_parts)]
    months_ts = pd.to_datetime(list(months))
    month_to_col = {m: m for m in months_ts}

    # Valid training versions: need t-12 >= first month AND t+5 <= last month.
    first_month, last_month = months_ts[0], months_ts[-1]
    valid_versions = [m for m in months_ts
                      if months_offset(m, -LAG_GI) >= first_month and months_offset(m, HORIZON - 1) <= last_month]

    gi_cols = [f'GI (t-{k})' for k in range(1, LAG_GI + 1)]
    mou_cols = [f'MOU_t-{k}' for k in range(1, LAG_MOU + 1)]
    scalar_cols = ['GI_cusum', 'MOU_Longest_High_Streak', 'MOU_YoY_Trend']
    y_cols = ['GI'] + [f'GI (t+{k})' for k in range(1, HORIZON)]
    feature_cols = gi_cols + scalar_cols + mou_cols

    def row_for(part, plant, v):
        gi_row = gi_plant_df.loc[(part, plant)]
        mou_row = mou_plant_df.loc[(part, plant)]
        gi_vals = [gi_row[months_offset(v, -k)] for k in range(1, LAG_GI + 1)]
        mou_vals = [mou_row[months_offset(v, -k)] for k in range(1, LAG_MOU + 1)]
        # scalar features: simple synthetic summaries
        cusum = float(np.sum(gi_vals))
        streak = float(np.max([np.sum(np.array(mou_vals) > 0.5), 0.0]))
        yoy = float(mou_vals[0] - mou_vals[-1])
        scalars = [cusum, streak, yoy]
        return gi_vals + scalars + mou_vals

    # --- training rows ---
    x_rows, y_rows, index_tuples = [], [], []
    for (part, plant) in gi_sel.index:
        for v in valid_versions:
            x_rows.append(row_for(part, plant, v))
            y_vals = [gi_plant_df.loc[(part, plant), months_offset(v, k)] for k in range(HORIZON)]
            y_rows.append(y_vals)
            index_tuples.append((part, plant, v))

    idx_train = pd.MultiIndex.from_tuples(index_tuples, names=['part', 'plant', 'version'])
    X_train = pd.DataFrame(x_rows, index=idx_train, columns=feature_cols, dtype=np.float64)
    y_train = pd.DataFrame(y_rows, index=idx_train, columns=y_cols, dtype=np.float64)

    # --- test rows (one per (part, plant) at version=test_month) ---
    v_test = pd.Timestamp(test_month + '-01')
    x_test_rows, test_index = [], []
    for (part, plant) in gi_sel.index:
        x_test_rows.append(row_for(part, plant, v_test))
        test_index.append((part, plant, v_test))
    idx_test = pd.MultiIndex.from_tuples(test_index, names=['part', 'plant', 'version'])
    X_test = pd.DataFrame(x_test_rows, index=idx_test, columns=feature_cols, dtype=np.float64)

    return gi_sel, X_train, y_train, X_test

# Peek at cluster 0
gi_sel0, X_tr0, y_tr0, X_te0 = build_cluster_frames(0, gi_plant_df, mou_plant_df, cluster_df, months, VALIDATION_MONTH)
print('gi_selected:', gi_sel0.shape, '| X_train:', X_tr0.shape, '| y_train:', y_tr0.shape, '| X_test:', X_te0.shape)
X_tr0.head(3)

## 4. Run the outer loop with DLModel

Mirrors `train.py` but with `DLModel` instead of `MLModel`.

In [ ]:
Model_result = pd.DataFrame(columns=['version', 'group', 'train_acc', 'val_acc', 'hyperparameter'])
Pred_plant = pd.DataFrame()

validation_month = [VALIDATION_MONTH]
plant_fab_mapping = {PLANT: 'FAB_A'}

for i in range(len(validation_month)):
    print('===', validation_month[i], '===')
    for c in cluster_df['cluster'].unique():
        print(f'--- cluster {c} ---')
        gi_selected, X_train, y_train, X_test = build_cluster_frames(
            c, gi_plant_df, mou_plant_df, cluster_df, months, validation_month[i]
        )

        pred_model, grid_params, cv_train, cv_validation = DLModel(
            df_gi=gi_selected,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            test_date=validation_month[i],
            validation_columns=VALIDATION_COLUMNS,
        )

        pred_plant = pred_model.reset_index()
        pred_plant['fab'] = pred_plant['plant'].map(plant_fab_mapping)
        pred_plant = pred_plant.set_index(keys=['version', 'fab', 'plant', 'part'])

        Model_result = pd.concat([
            Model_result,
            pd.DataFrame([{'version': validation_month[i], 'group': str(c),
                           'train_acc': cv_train, 'val_acc': cv_validation,
                           'hyperparameter': grid_params}])
        ], ignore_index=True)

        print('Train / Val score:', cv_train, cv_validation)
        Pred_plant = pd.concat([Pred_plant, pred_plant])

Pred_plant['model'] = 'UsageBased'

## 5. Inspect results

In [ ]:
print('Model_result:')
display(Model_result)
print('\nPred_plant head:')
display(Pred_plant.head(10))
print('\nPred_plant shape:', Pred_plant.shape)
print('Pred_plant describe:')
display(Pred_plant.describe())